# Stochastic Simulation — Exam Preparation Guide
### MATH60047 (non-mastery) | Based on MATH70047 past exams 2023–2025

---

## ⚠️ YOUR EXAM FORMAT — CRITICAL

> **You answer 3 of 4 questions. Each question = 20 marks. Total = 60 marks.**
>
> The exam paper has 4 questions (the paper is shared with MATH70047 mastery students who answer all 4). As a MATH60047 non-mastery student, you answer any 3.
>
> **Recommended strategy: answer Q1 + Q2 + Q3. Skip Q4 (SMC/particle filtering).**

| Question | Topic | Your plan |
|----------|-------|-----------|
| **Q1** | Exact sampling: inversion/CDF proofs, 2D transforms, rejection sampling | ✅ Do this |
| **Q2** | MC/IS: estimator variance, optimise proposal, SNIS | ✅ Do this |
| **Q3** | MCMC + Bayesian inference | ✅ Do this |
| **Q4** | Sequential Monte Carlo / Particle Filtering | ⚡ Optional backup only |

---

## Exam File Status — All Fixed

| File | Actual Year | Module Code | Status |
|------|-------------|-------------|--------|
| `2020.txt` | 2020 | MATH97085 | ❌ OLD curriculum — skip |
| `2021.txt` | 2021 | MATH97085 | ❌ OLD curriculum — skip |
| `2022.txt` | 2022 | MATH60047/97085 | ⚠️ Transitional (mostly skip) |
| `2023.txt` | **2023** | MATH70047 | ✅ **CURRENT** |
| `2024.txt` | **2024** | MATH70047 | ✅ **CURRENT** |
| `2025.txt` | **2025** | MATH70047 | ✅ **CURRENT** |

**Study only 2023, 2024, 2025.** The 2020/2021 exams test entirely different content (PRNGs, LCGs, chi-squared tests, ratio-of-uniforms, antithetic variates, ARS/ARMS) that is completely absent from the current MATH60047/70047 lecture notes and has not appeared since 2022.

---

## How Q3/Q4 Evolved Across MATH70047 Years

| Year | Q3 Topics | Q4 Topics |
|------|-----------|-----------|
| **2023** | Gibbs (3D), MH (indep+symm), MALA, IG-Gaussian conjugacy | SISR, BPF, Particle MH |
| **2024** | MH+MALA+ULA+SG-ULA, Gamma-Poisson sequential Bayes | SIS weight derivation, variance degeneracy proof |
| **2025** | MH (indep+general), detailed balance proof, SA+Langevin optimisation, ULA AR(1) | BPF+log-likelihood, gradient ascent + SIR smoother |

---

## Study Priority

1. **Q3 (MCMC)** — pseudocodes + proofs, highest marks-per-effort, consistently most complex
2. **Q2 (IS/MC)** — variance computations and optimisation, well-defined techniques
3. **Q1 (Sampling)** — very algorithmic, standard CDF proofs and rejection analysis
4. **Q4 (SMC)** — ⚡ optional: only if comfortable, use as backup if Q3 looks hard

# Q1 — Exact Sampling Methods
## Topic Pattern (MATH70047 — 2023, 2024 & 2025)

**Part (a):** Prove inversion sampler / derive CDF / general transform  
**Part (b):** 2D transformation (Box-Muller / chi-squared polar) — derive joint density  
**Part (c):** Rejection sampling — pseudocode, acceptance rate, constraint  
**Part (d):** Optimise rejection sampler (find optimal proposal parameter)

---

## 1.1 The Inversion Sampler

**Standard version:** Set $Y = F_Y^{-1}(U)$ where $U \sim \text{Unif}(0,1)$.

**Standard proof (2024 Q1a — "using only the transformation formula"):**

$g(u) = F_Y^{-1}(u)$, so $g^{-1}(y) = F_Y(y)$, $|dg^{-1}/dy| = p_Y(y)$, $p_U(F_Y(y)) = 1$:
$$p_Y(y) = p_U(g^{-1}(y))\left|\frac{dg^{-1}}{dy}\right| = 1 \cdot p_Y(y) \checkmark$$

**General transform version (2025 Q1a):** To transform $X \sim p_X$ into $Y \sim p_Y$, use $g(x) = F_Y^{-1}(F_X(x))$.

**Proof (CDF argument):**
$$P(Y \leq y) = P(F_Y^{-1}(F_X(X)) \leq y) = P(F_X(X) \leq F_Y(y)) = F_Y(y) \checkmark$$

---

## 1.2 CDF Derivation by Substitution (2023 Q1a — new technique)

**Rayleigh distribution:** $p(x) = \frac{x}{\sigma^2}e^{-x^2/(2\sigma^2)},\ x \geq 0$.

**Derive $F_X(x)$:** Substitute $u = y^2$, $du = 2y\,dy$:
$$F_X(x) = \int_0^x \frac{y}{\sigma^2}e^{-y^2/(2\sigma^2)}dy = \int_0^{x^2} \frac{1}{2\sigma^2}e^{-u/(2\sigma^2)}du = 1 - e^{-x^2/(2\sigma^2)}$$

**Inversion:** $1 - e^{-x^2/(2\sigma^2)} = u \implies x = \sqrt{-2\sigma^2\log(1-u)}$

**Sampler:** $X = \sqrt{-2\sigma^2 \log U}$ works since $U$ and $1-U$ have the same distribution.

> **Exam tip:** Any CDF of form $F(x) = 1 - e^{-h(x)}$ can be inverted by: $h(x) = -\log(1-u)$.

---

## 1.3 Rejection Sampling — General Pseudocode

```
Given target p(x), proposal q(x), bound M s.t. p(x) ≤ Mq(x) for all x:
1. Sample Y ~ q(x)
2. Sample U ~ Unif(0, 1)
3. If U ≤ p(Y) / (M·q(Y)): accept, output Y
4. Else: reject, go to 1
```

**Optimal M:** $M = \sup_x \frac{p(x)}{q(x)}$ (or $\frac{\bar{p}(x)}{q(x)}$ for unnormalised $\bar{p}$)

---

## 1.4 Acceptance Rate (always asked, 3 marks)

$$\hat{a} = \int \frac{p(y)}{Mq(y)} q(y)\, dy = \frac{1}{M}$$

**With unnormalised target:** $\hat{a} = Z/\bar{M}$ where $\bar{M} = \sup_x \bar{p}(x)/q(x)$

---

## 1.5 Optimising the Acceptance Rate

**Recipe:**
1. Find $x^* = \arg\max_x R(x)$ where $R(x) = \bar{p}(x)/q_\lambda(x)$ — set $\frac{d}{dx}\log R(x) = 0$
2. $M_\lambda = R(x^*)$ — bound as function of $\lambda$
3. Minimise $M_\lambda$: set $\frac{d}{d\lambda}\log M_\lambda = 0$

---

### 2023 Q1b — Rejection with Gaussian Proposal (Rayleigh target)

**Target:** $p(x) = \frac{x}{\sigma^2}e^{-x^2/(2\sigma^2)}$, $x \geq 0$. **Proposal:** $q_\mu(x) = \mathcal{N}(x; \mu, \sigma^2)$.

**Step 1 — Ratio:**
$$R(x) = \frac{p(x)}{q_\mu(x)} = \frac{\sqrt{2\pi}\,x}{\sigma}\exp\!\left(\frac{\mu^2}{2\sigma^2} - \frac{\mu x}{\sigma^2}\right)$$

> Constraint: need $\mu > 0$ for $R(x) \to 0$ as $x \to \infty$ (otherwise $e^{-\mu x/\sigma^2}$ grows).

**Step 2 — Maximise:** $\frac{d\log R}{dx} = \frac{1}{x} - \frac{\mu}{\sigma^2} = 0 \implies x^* = \frac{\sigma^2}{\mu}$

$$M_\mu = \frac{\sqrt{2\pi}\,\sigma}{\mu}\exp\!\left(\frac{\mu^2}{2\sigma^2} - 1\right)$$

**Step 3 — Optimise $\mu$:** $\frac{d\log M_\mu}{d\mu} = -\frac{1}{\mu} + \frac{\mu}{\sigma^2} = 0 \implies \boxed{\mu^* = \sigma}$

$$M_{\mu^*} = \sqrt{2\pi/e}, \quad \hat{a} = \sqrt{e/(2\pi)}$$

**Pseudocode for optimal sampler:**
```
1. Sample X' ~ N(x; σ, σ²)     [Gaussian centred at σ]
2. Sample U ~ Unif(0,1)
3. Accept if U ≤ (X'/σ) · exp(−X'/σ + 1)
4. Else reject and go to 1
```

> **Compare to 2024:** Erlang target with Exp proposal → $\mu^* = \lambda/k$. Here, Rayleigh target with Gaussian proposal → $\mu^* = \sigma$. Same recipe, different algebra.

---

### 2025 Q1c — Exponential Proposal

$\bar{p}(x) = x^2 e^{-x/2}$, $q_\lambda(x) = \lambda e^{-\lambda x}$, $x > 0$.

$x^* = \frac{4}{1-2\lambda}$ (constraint: $\lambda < 1/2$), $\quad M_\lambda = \frac{16}{\lambda(1-2\lambda)^2}e^{-2}$

$\frac{d\log M_\lambda}{d\lambda} = -\frac{1}{\lambda} + \frac{4}{1-2\lambda} = 0 \implies \boxed{\lambda^* = 1/6}$

Acceptance rate: $\hat{a} = Z/M_{1/6}$, where $Z = \int_0^\infty x^2e^{-x/2}dx = 16$.

### 2024 Q1d — Erlang Target with Exponential Proposal

$p(x) = \frac{x^{k-1}e^{-\lambda x}}{(k-1)!}$, $q_\mu(x) = \mu e^{-\mu x}$, $\mu < \lambda$.

$x^* = (k-1)/(\lambda-\mu)$, $\quad \frac{d\log M_\mu}{d\mu} = 0 \implies \boxed{\mu^* = \lambda/k}$

---

## 1.6 Distribution of Number of Accepted Samples (2025 Q1c iv–v)

Each iteration: independent Bernoulli($\hat{a}$). For $K$ total iterations: $N \sim \text{Binomial}(K, \hat{a})$.

$$E[N] = K\hat{a}, \quad P(N=n) = \binom{K}{n}\hat{a}^n(1-\hat{a})^{K-n}$$

---

## 1.7 Multivariate (2D) Transformation

**Box-Muller (2024 Q1b):** $U_1 \sim \text{Unif}(0,1)$, $U_2 \sim \text{Unif}(-\pi,\pi)$, $X_1 = \cos(U_2)\sqrt{-2\log U_1}$:

Jacobian $\det J_{g^{-1}} = -e^{-(x_1^2+x_2^2)/2}$, giving $p_{X_1,X_2} = \mathcal{N}(0,1)\mathcal{N}(0,1)$.

**Chi-squared Polar (2025 Q1b):** $r \sim \chi^2(k)$, $\theta \sim \text{Unif}(-\pi,\pi)$, $X=\sqrt{r}\cos\theta$:

$|\det J| = 2$, $p_{X,Y}(x,y) = \frac{1}{2^{k/2}\Gamma(k/2)\pi}(x^2+y^2)^{k/2-1}e^{-(x^2+y^2)/2}$

Special case $k=2$ gives Box-Muller since $\chi^2(2) = \text{Exp}(1/2)$.

# Q2 — Monte Carlo Integration & Importance Sampling
## Topic Pattern (MATH70047 — 2023, 2024 & 2025)

**2023:** MC/IS for marginal likelihood $p(y_0)$ → optimal IS proposal = posterior  
**2024:** IS variance (Rayleigh proposal) → optimise $\lambda$ → compare IS vs MC  
**2025:** SNIS for marginal likelihood → gradient identity proof → IS with $\mathcal{N}(\mu,1/2)$ for indicator

---

## 2.1 The Core Goal

Estimate $\bar{\varphi} = E_{p}[\varphi(X)] = \int \varphi(x)p(x)dx$ using samples from a tractable proposal $q$.

---

## 2.2 Plain Monte Carlo Estimator

$$\hat{\varphi}^N_{MC} = \frac{1}{N}\sum_{i=1}^N \varphi(X_i), \quad X_i \stackrel{iid}{\sim} p$$

$$\text{Var}(\hat{\varphi}^N_{MC}) = \frac{1}{N}\left(E_p[\varphi^2] - (E_p[\varphi])^2\right)$$

**Error metrics (Ch3 §3.2):** bias $= E[\hat{\varphi}^N] - \bar{\varphi}$; variance; MSE $=$ bias$^2$ + var; RMSE; RAE $= |\hat{\varphi}^N - \bar{\varphi}|/|\bar{\varphi}|$. MC estimator is **unbiased** → MSE = variance. Convergence rate: $O(1/\sqrt{N})$.

---

## 2.3 Importance Sampling (IS) Estimator

$$\hat{\varphi}^N_{IS} = \frac{1}{N}\sum_{i=1}^N \varphi(X_i) w(X_i), \quad w(x) = \frac{p(x)}{q(x)}, \quad X_i \sim q$$

$$\text{Var}(\hat{\varphi}^N_{IS}) = \frac{1}{N}\left(\int \frac{\varphi(x)^2 p(x)^2}{q(x)}dx - \bar{\varphi}^2\right)$$

IS is **unbiased**: $E_q[w(X)\varphi(X)] = \int \frac{p(x)}{q(x)}\varphi(x)q(x)dx = E_p[\varphi(X)]$.

**Finiteness condition:** IS variance is finite iff $\int \frac{p^2(x)\varphi^2(x)}{q(x)}dx < \infty$. A sufficient condition: $q(x) > 0$ whenever $p(x)\varphi(x) \neq 0$ (full support), and the weight ratio is bounded.

---

## 2.4 MC and IS for Marginal Likelihood (2023 Q2 — new type)

**Setting:** $p(y_0) = \int p(y_0|x)p(x)dx$ — estimate this integral.

Test function: $\varphi(x) = p(y_0|x)$ (the likelihood).

**MC estimator** (sample from prior $p(x)$):
$$\hat{p}^N_{MC}(y_0) = \frac{1}{N}\sum_{i=1}^N p(y_0|X_i), \quad X_i \stackrel{iid}{\sim} p(x)$$

This is **unbiased**: $E[\hat{p}^N_{MC}(y_0)] = p(y_0)$.

Variance: $\text{Var}(\hat{p}^N_{MC}) = \frac{1}{N}\left(\int p^2(y_0|x)p(x)dx - p(y_0)^2\right)$

**Failure mode:** If $y_0$ is very unlikely (small $p(y_0)$), the prior $p(x)$ concentrates away from the region where $p(y_0|x)$ is large → MC gives very noisy estimates or near-zero.

**IS estimator** (sample from proposal $q(x)$):
$$\hat{p}^N_{IS}(y_0) = \frac{1}{N}\sum_{i=1}^N p(y_0|X_i)\,\frac{p(X_i)}{q(X_i)}, \quad X_i \sim q(x)$$

IS variance finiteness condition: $q(x) > 0$ whenever $p(y_0|x)p(x) > 0$.

---

## 2.5 Optimal IS Proposal = Posterior (2023 Q2c — key insight)

**Model:** $p(x) = \mathcal{N}(x;0,1)$, $p(y_0|x) = \mathcal{N}(y_0;x,1)$, $q_\mu(x) = \mathcal{N}(x;\mu,1/2)$.

**Goal:** Choose $\mu$ to minimise IS variance $V(\mu) = E_{q_\mu}\!\left[\frac{p^2(y_0|X)p^2(X)}{q_\mu^2(X)}\right]$.

**Key insight:** The IS estimator has **zero variance** when $q(x) \propto p(y_0|x)p(x) = p(x|y_0)\cdot p(y_0)$, i.e., when $q$ is the **posterior** $p(x|y_0)$.

**Why?** Then $w(x)p(y_0|x) = \frac{p(x)}{q_\mu(x)}p(y_0|x) = \frac{p(x,y_0)}{p(x|y_0)} = p(y_0)$ = constant → zero variance.

**Computation of $\mu^*$:** The posterior under this Gaussian model is:
$$p(x|y_0) \propto e^{-(y_0-x)^2/2}\cdot e^{-x^2/2} = e^{-x^2 + y_0 x - y_0^2/2} \propto \mathcal{N}(x;\,y_0/2,\, 1/2)$$

Since $q_\mu = \mathcal{N}(x;\mu,1/2)$ can match the posterior exactly: $\boxed{\mu^* = y_0/2}$.

**Full derivation of $V(\mu)$** (completing the square):

$$V(\mu) = \int \frac{e^{-(y_0-x)^2}}{2\pi}\cdot\frac{e^{-x^2}}{2\pi}\cdot\frac{\sqrt{\pi}\,e^{(x-\mu)^2}}{1}\,dx = \frac{1}{4\pi}e^{\mu^2 - 2y_0\mu}$$

$$\frac{d\log V}{d\mu} = 2\mu - 2y_0 = 0 \implies \mu^* = y_0/2 \checkmark$$

> **Why IS works when MC fails:** IS with $q_{\mu^*=y_0/2}$ concentrates samples near the posterior mean, directly in the region where $p(y_0|x)$ is large, even when $y_0$ has tiny prior probability.

---

## 2.6 Computing IS Variance — The Standard Technique (2024 Q2)

**Setup:** $p(x) = xe^{-x^2/2}$ (Rayleigh), $\varphi(x) = x$, proposal $q_\lambda(x) = \frac{x}{\lambda^2}e^{-x^2/(2\lambda^2)}$

$$V = \int_0^\infty \frac{x^2 p^2(x)}{q_\lambda(x)}dx = \lambda^2\int_0^\infty x^3 e^{-x^2(1-1/(2\lambda^2))}dx = \frac{2\lambda^6}{(2\lambda^2-1)^2}$$

$$\text{Var}(\hat{\varphi}^N_{IS}) = \frac{1}{N}\left(\frac{2\lambda^6}{(2\lambda^2-1)^2} - \frac{\pi}{2}\right)$$

Optimise: $\boxed{\lambda^* = \sqrt{3/2}}$

**IS vs MC:** IS can outperform MC even with i.i.d. samples from $p$ available, by concentrating on high-$\varphi(x)$ regions.

---

## 2.7 Self-Normalised IS (SNIS) for Unnormalised Targets (2025 Q2a)

When only $\bar{p}(x) \propto p(x)$ is known:

$$\hat{\varphi}^N_{SNIS} = \frac{\sum_{i=1}^N \varphi(X_i)W_i}{\sum_{i=1}^N W_i}, \quad W_i = \frac{\bar{p}(X_i)}{q(X_i)}, \quad X_i \sim q$$

**Biased** but consistent. Bias is $O(1/N)$, MSE is $O(1/N)$.

**Important distinction (Prop 3.6 in lecture notes):**  
The unnormalised marginal likelihood estimator $\hat{p}^N(y) = \frac{1}{N}\sum_{i=1}^N W_i$ is **unbiased** for $p(y)$, even when SNIS for posterior expectations is biased.

---

## 2.8 Optimising IS for Indicator Function (2025 Q2c)

$p(x) = \mathcal{N}(x;0,1)$, $\varphi(x) = \mathbf{1}_{x \geq K}$, proposal $q_\mu(x) = \mathcal{N}(x;\mu,1/2)$.

After simplification:
$$V(\mu) = \frac{e^{\mu^2 - 2\mu K}}{4\mu\sqrt{\pi}}, \quad \mu > 0$$

$\frac{d\log V}{d\mu} = 2\mu - 2K - \frac{1}{\mu} = 0 \implies 2\mu^2 - 2K\mu - 1 = 0$

$$\boxed{\mu^* = \frac{K + \sqrt{K^2+2}}{2}}$$

> **Pattern match with 2023 Q2c:** Both find optimal proposal for IS in a Gaussian model, both get $\mu^*$ from a clean calculus argument. The difference: 2023 gives $\mu^* = y_0/2$ (linear, because posterior is exactly the parametric family); 2025 gives a quadratic formula (proposal family doesn't match posterior exactly).

---

## 2.9 Gradient Estimation / Fisher's Identity (2025 Q2b, Ch5 §5.1.2)

$$\nabla_\theta \log p_\theta(y) = E_{p_\theta(x|y)}[\nabla_\theta \log p_\theta(x,y)]$$

**Proof:**
$$\frac{\nabla_\theta p_\theta(y)}{p_\theta(y)} = \frac{\int \nabla_\theta p_\theta(x,y)dx}{p_\theta(y)} = \frac{\int p_\theta(x,y)\nabla_\theta \log p_\theta(x,y)dx}{p_\theta(y)} = E_{p_\theta(x|y)}[\nabla_\theta \log p_\theta(x,y)]$$

This identity links IS (estimate the gradient using samples) to the SOUL algorithm (Q3 cell) and the EM algorithm. The first step uses $\nabla \log f = \nabla f / f$ (log-derivative trick).

---

## 2.10 Effective Sample Size (ESS) Diagnostic (Ch3 §3.5.3)

$$\boxed{\text{ESS}_N = \frac{1}{\sum_{i=1}^N \bar{w}_i^2}}$$

where $\bar{w}_i = W_i / \sum_j W_j$ are **normalised** SNIS weights (so $\sum \bar{w}_i = 1$).

| Scenario | ESS value | Interpretation |
|----------|-----------|---------------|
| All weights equal: $\bar{w}_i = 1/N$ | $\text{ESS} = N$ | Perfectly efficient, equivalent to i.i.d. samples |
| All weight on one particle: $\bar{w}_j = 1$, rest 0 | $\text{ESS} = 1$ | Complete degeneracy |
| General case | $1 \leq \text{ESS} \leq N$ | Partial efficiency |

**Diagnostic use:** If ESS/N is very small (e.g. $< 0.1$), the proposal $q$ is a poor match to the target — choose a better proposal or use more samples.

---

## 2.11 Sampling Importance Resampling (SIR) (Ch3 §3.5.2)

Given a weighted sample $\{(X_i, \bar{w}_i)\}_{i=1}^N$ from SNIS, convert to unweighted samples approximately distributed as $p^*(x)$:

**Algorithm:**
1. Draw index $k \sim \text{Discrete}(\bar{w}_1, \ldots, \bar{w}_N)$ — multinomial over normalised weights
2. Set $\bar{X} = X_k$
3. Repeat $N$ times to get $\bar{X}_1, \ldots, \bar{X}_N$

**Result:** $\bar{X}_i \approx p^*(x)$ (approximately i.i.d.), but with extra variance vs the weighted sample.

> This is identical to the resampling step inside the BPF (Q4). Knowing SIR lets you explain BPF resampling if asked in Q2 context.

# Q3 — MCMC & Bayesian Inference
## Topic Pattern (MATH70047 — 2023, 2024 & 2025)

**2023:** Gibbs sampler (3D) + MH pseudocodes + MALA + IG-Gaussian conjugacy  
**2024:** MH pseudocodes → MALA → ULA → stochastic gradient → Gamma-Poisson conjugacy  
**2025:** Independent MH → detailed balance proof → SA + Langevin optimisation → ULA AR(1)

---

## 3.1 Detailed Balance — Definition and Proof

**Definition (2023 Q3a — always state this first):**

A kernel $K(x'|x)$ satisfies **detailed balance** with respect to $p^*(x)$ if:
$$p^*(x)\,K(x'|x) = p^*(x')\,K(x|x')$$

**Why it matters:** Detailed balance $\implies$ $p^*$ is a stationary distribution. (The converse is not always true — global balance is weaker.)

**Detailed balance PROOF for MH (2025 Q3a iv — 5 marks):**

Let $\pi(x'|x) = q(x'|x)\alpha(x,x')$. Need to show $p^*(x)\pi(x'|x) = p^*(x')\pi(x|x')$.

**WLOG Case 1:** Assume $p^*(x')q(x|x') \geq p^*(x)q(x'|x)$, so $\alpha(x,x') = \frac{p^*(x')q(x|x')}{p^*(x)q(x'|x)}$, $\alpha(x',x) = 1$.

$$\text{LHS} = p^*(x)\cdot q(x'|x)\cdot\frac{p^*(x')q(x|x')}{p^*(x)q(x'|x)} = p^*(x')q(x|x') = p^*(x')\cdot q(x|x')\cdot 1 = \text{RHS} \checkmark$$

**Case 2:** Symmetric — swap roles of $x$ and $x'$, identical computation. ✓

---

## 3.2 Gibbs Sampler (2023 Q3a — only in 2023)

**3D target $p(x,y,z)$** — update each component given the others:

```
Given current state (x_{n-1}, y_{n-1}, z_{n-1}):
  1. Sample x_n  ~ p(x | y_{n-1}, z_{n-1})
  2. Sample y_n  ~ p(y | x_n, z_{n-1})
  3. Sample z_n  ~ p(z | x_n, y_n)
```

> Updates can be in any order. Each step uses the most recent values of ALL other components.

**Note on marks:** The order of updates matters — show you use $x_n$ (not $x_{n-1}$) in step 2 and $x_n, y_n$ in step 3.

---

## 3.3 Metropolis-within-Gibbs (2023 Q3a iii)

**When to use:** If full conditional distributions are not available in closed form (e.g., only unnormalised $\bar{p}(x|y,z)$ known).

**Algorithm:** Replace each Gibbs update with a Metropolis accept/reject step:

```
Given current state (x_{n-1}, y_{n-1}, z_{n-1}):
  1. Run MH step targeting p̄(x | y_{n-1}, z_{n-1}) → get x_n
  2. Run MH step targeting p̄(y | x_n, z_{n-1})    → get y_n
  3. Run MH step targeting p̄(z | x_n, y_n)         → get z_n
```

Each MH step uses an arbitrary proposal (e.g., $\mathcal{N}$ random walk) with accept/reject ratio using the unnormalised full conditional.

---

## 3.4 MCMC Diagnostics (2023 Q3a iv)

**Thinning:** Keep only every $k$-th sample to reduce autocorrelation (wastes computation but gives less correlated draws).

**Effective Sample Size (ESS):** $\text{ESS} = N / (1 + 2\sum_{k=1}^\infty \rho_k)$ where $\rho_k$ = lag-$k$ autocorrelation. Measures how many i.i.d. samples the correlated chain is equivalent to.

---

## 3.5 Metropolis-Hastings (MH) — General Version

**Pseudocode (memorise exactly):**
```
Target: p*(x). Proposal: q(x'|x). Initialise X_0.
For k = 0, 1, 2, ...:
  1. Propose: X' ~ q(x'|X_k)
  2. α = min(1, [p*(X')·q(X_k|X')] / [p*(X_k)·q(X'|X_k)])
  3. X_{k+1} = X' with prob α; else X_{k+1} = X_k
```

---

## 3.6 MH — Independent Proposal (2023 Q3b i, 2025 Q3a i)

$q(x'|x) = q(x')$:
$$\alpha(x, x') = \min\!\left(1,\ \frac{p^*(x')q(x)}{p^*(x)q(x')}\right)$$

**Rejection probability at $x$** (2025 Q3a ii): $r(x) = 1 - \int q(x')\,\alpha(x, x')\,dx'$ (depends on $x$).

---

## 3.7 MH — Symmetric Proposal (2023 Q3b ii, 2024)

$q(x'|x) = q(x|x')$ → $q$ terms cancel:
$$\alpha(x, x') = \min\!\left(1,\ \frac{p^*(x')}{p^*(x)}\right)$$

---

## 3.8 Metropolis-Adjusted Langevin Algorithm (MALA) (2023 Q3b iii, 2024)

For $p^*(x) = \mathcal{N}(x;\mu,\sigma^2)$: gradient $\nabla\log p^*(x) = -(x-\mu)/\sigma^2$.

```
Propose: X' = X_k - γ(X_k-μ)/σ² + √(2γ) W_k,  W_k ~ N(0,1)
where q(x'|x) = N(x'; x - γ(x-μ)/σ², 2γ)
Accept with α = min(1, [p*(X')·q(X_k|X')] / [p*(X_k)·q(X'|X_k)])
```

**Skipping the MH step** → ULA → chain has a **different** (biased) stationary distribution (not exactly $p^*$).

> **2023 exam phrasing:** "What happens if we skip the Metropolis step?" → Answer: obtain ULA → targets $p^*$ only approximately (bias).

---

## 3.9 Unadjusted Langevin Algorithm (ULA) (2024, 2025)

$$X_{k+1} = X_k + \gamma \nabla \log p^*(X_k) + \sqrt{2\gamma}\, W_k, \quad W_k \sim \mathcal{N}(0,1)$$

No accept/reject → biased. For Bayesian posterior: $\nabla \log p^*(x) = \nabla \log p(y|x) + \nabla \log p(x)$.

**Stochastic gradient ULA (2024):** Replace full gradient with minibatch gradient for large $n$.

---

## 3.10 Limiting Distribution of Modified ULA (2025 Q3c)

For $p^*(x) = \mathcal{N}(x;\mu,\sigma^2)$, ULA gives AR(1): $X_{k+1} = \rho X_k + \frac{\gamma\mu}{\sigma^2} + \sigma_q W_k$, $\rho = 1-\gamma/\sigma^2$

Stationary distribution: $\mathcal{N}\!\left(\mu,\,\frac{\sigma_q^2}{1-\rho^2}\right)$

**To make ULA unbiased:** set $\sigma_q^2 = \sigma^2(1-\rho^2) = 2\gamma - \gamma^2/\sigma^2$

---

## 3.11 Global Optimisation Algorithms (2025 Q3b)

**Simulated Annealing:** MH with temperature $T_k \to 0$. High $T$ = random walk (exploration); low $T$ = greedy (exploitation). Schedule $T_k \to 0$ allows escaping local optima early.

**Langevin for Optimisation:** ULA with $\gamma_k \to 0$: $x_{k+1} = x_k + \gamma_k\nabla f(x_k) + \sqrt{2\gamma_k}W_k$. Noise enables escaping local optima; $\gamma_k \to 0$ eliminates noise over time → converges to optimum.

---

## 3.12 Bayesian Conjugate Updates

**Gamma-Poisson (2024 Q3b):** Prior $\text{Gamma}(\alpha,\beta)$ + likelihood $\text{Poisson}$:
$$p(x|y) = \text{Gamma}(\alpha+y,\, \beta+1)$$

**Sequential update proof:** $p(x|y_{1:n}) \propto p(y_n|x)\,p(x|y_{1:n-1})$ follows from:
$$p(y_{1:n}|x) = \prod_{i=1}^n p(y_i|x) \implies p(x|y_{1:n}) \propto p(y_n|x)\cdot\underbrace{p(y_{1:n-1}|x)p(x)}_{\propto p(x|y_{1:n-1})}$$

Gamma-Poisson recursion: $\alpha_n = \alpha_{n-1} + y_n$, $\beta_n = \beta_{n-1} + 1$.

---

## 3.13 Inverse-Gamma / Gaussian Conjugacy (2023 Q3c — only in 2023)

**Model:** Prior $\sigma^2 \sim \text{IG}(\alpha,\beta)$, i.e., $p(\sigma^2) \propto (\sigma^2)^{-\alpha-1}e^{-\beta/\sigma^2}$

Likelihood: $x|\sigma^2 \sim \mathcal{N}(0,\sigma^2)$, i.e., $p(x|\sigma^2) \propto (\sigma^2)^{-1/2}e^{-x^2/(2\sigma^2)}$

**Posterior (derive from scratch — 3 marks):**
$$p(\sigma^2|x) \propto (\sigma^2)^{-(\alpha+1/2)-1}\exp\!\left(-\frac{\beta+x^2/2}{\sigma^2}\right) = \text{IG}\!\left(\alpha+\tfrac{1}{2},\, \beta+\tfrac{x^2}{2}\right)$$

**Gibbs sampler for the (x, σ²) model:**
```
Given state (x_{n-1}, σ²_{n-1}):
  1. Sample x_n        ~ N(0, σ²_{n-1})
  2. Sample σ²_n       ~ IG(α + 1/2, β + x_n²/2)
```

> **Key step in derivation:** Combine exponents $-x^2/(2\sigma^2)$ and $-\beta/\sigma^2$ into $-(\beta+x^2/2)/\sigma^2$, and combine powers $-1/2$ and $-\alpha-1$ into $-(\alpha+1/2)-1$.

> **Why this is Gibbs:** Both full conditionals $p(\sigma^2|x)$ and $p(x|\sigma^2)$ are available in closed form (IG and Gaussian respectively) → can sample directly → no MH needed.

# Q4 — Sequential Monte Carlo (SMC) & Particle Filtering
## ⚡ OPTIONAL FOR MATH60047 — Only prepare if you need a backup question

> **You need to answer 3 of 4 questions.** Recommended: Q1 + Q2 + Q3.  
> Only study Q4 if you have spare time, or if Q3 looks hard on exam day and you want a fallback.

## Topic Pattern (MATH70047 — 2023, 2024 & 2025)

**2023:** Filtering recursions → SISR → BPF → Particle Metropolis-Hastings (pMH)  
**2024:** Joint density derivation → SIS weight recursion derivation → variance degeneracy proof  
**2025:** BPF + log marginal likelihood → gradient of log-likelihood → gradient ascent + SIR smoother

---

## 4.1 State Space Model

$$X_0 \sim \pi_0(x_0), \quad X_k|X_{k-1} \sim f(x_k|x_{k-1}), \quad Y_k|X_k \sim g(y_k|x_k)$$

**Joint density (2023 Q4a, 2024 Q4a):**
$$p(x_{0:T}, y_{1:T}) = \pi_0(x_0)\prod_{k=1}^T f(x_k|x_{k-1})\,g(y_k|x_k)$$

---

## 4.2 Filtering Recursions (2023 Q4a iii — 3 marks)

Given the **filtering distribution** $p(x_{n-1}|y_{1:n-1})$ at time $n-1$:

**Step 1 — Prediction** (marginalise transition):
$$p(x_n|y_{1:n-1}) = \int f(x_n|x_{n-1})\,p(x_{n-1}|y_{1:n-1})\,dx_{n-1}$$

**Step 2 — Update** (Bayes' rule with new observation $y_n$):
$$p(x_n|y_{1:n}) = \frac{g(y_n|x_n)\,p(x_n|y_{1:n-1})}{p(y_n|y_{1:n-1})}$$

where $p(y_n|y_{1:n-1}) = \int g(y_n|x_n)\,p(x_n|y_{1:n-1})\,dx_n$ is the **predictive likelihood**.

> These recursions are exact but intractable in general — motivates particle methods.

---

## 4.3 SISR — Sequential Importance Sampling with Resampling (2023 Q4b)

The core algorithm that combines SIS with resampling at each step to avoid weight degeneracy:

```
Initialise: X_0^{(i)} ~ π_0, W_0^{(i)} = 1/N  for i=1,...,N
For t = 1, ..., T:
  1. Propagate:  X̄_t^{(i)} ~ q(x_t | X_{t-1}^{(i)})  for each i
  2. Weight:     W_t^{(i)} ∝ f(X̄_t^{(i)}|X_{t-1}^{(i)}) · g(y_t|X̄_t^{(i)}) / q(X̄_t^{(i)}|X_{t-1}^{(i)})
                 Normalise: w_t^{(i)} = W_t^{(i)} / Σ_j W_t^{(j)}
  3. Resample:   Draw indices i_k ~ Discrete(w_1,...,w_N); set X_t^{(k)} = X̄_t^{(i_k)}
```

**SIS vs SISR:** SIS = no resampling step → weights accumulate → weight degeneracy over time. SISR = resample at each step → weights reset to $1/N$ → avoids degeneracy.

---

## 4.4 Bootstrap Particle Filter (BPF)

**Special case of SISR** with $q = f$ (transition as proposal):

```
For t = 1, ..., T:
  1. Resample X_{t-1}^{(i)} with weights {W_{t-1}^{(i)}}
  2. Propagate: X_t^{(i)} ~ f(x_t | X_{t-1}^{(i)})
  3. Weight:    w_t^{(i)} = g(y_t | X_t^{(i)})   [f cancels in ratio]
  4. Normalise: W_t^{(i)} = w_t^{(i)} / Σ_j w_t^{(j)}
```

**Key:** Using $q=f$ makes the weight independent of path history (only depends on $g$ at current step).

---

## 4.5 Marginal Likelihood Estimator from BPF (2023 Q4c i)

At each step $t$, **before normalising**, the mean unnormalised weight estimates the predictive likelihood:

$$\hat{p}^N(y_t | y_{1:t-1}) = \frac{1}{N}\sum_{i=1}^N w_t^{(i)}$$

**Overall marginal likelihood estimator:**
$$\hat{p}^N_\theta(y_{1:T}) = \prod_{t=1}^T \frac{1}{N}\sum_{i=1}^N w_t^{(i)}$$

**Log-space computation (numerical stability):**
$$\log\hat{p}^N_\theta(y_{1:T}) = \sum_{t=1}^T \log\!\left(\frac{1}{N}\sum_{i=1}^N w_t^{(i)}\right)$$

Use **log-sum-exp** trick: $\log\sum_i e^{a_i} = \max_j a_j + \log\sum_i e^{a_i - \max_j a_j}$

---

## 4.6 Particle Metropolis-Hastings (pMH) (2023 Q4c ii — 4 marks, only in 2023)

**Goal:** Sample from $p(\theta|y_{1:T})$ when marginal likelihood $p_\theta(y_{1:T})$ is intractable.

**Key property:** BPF gives an **unbiased** estimate of $p_\theta(y_{1:T})$, so substituting it into MH still targets the correct posterior.

**Pseudocode:**
```
Initialise θ_0. Run BPF with θ_0 → get p̂_{θ_0}(y_{1:T}).
For k = 0, 1, 2, ...:
  1. Propose θ* ~ q(θ | θ_k)
  2. Run BPF with θ* → get p̂_{θ*}(y_{1:T})
  3. α = min(1, [p̂_{θ*}(y_{1:T}) · p(θ*) · q(θ_k|θ*)] / [p̂_{θ_k}(y_{1:T}) · p(θ_k) · q(θ*|θ_k)])
  4. With prob α: θ_{k+1} = θ*,  update stored p̂
     Else:        θ_{k+1} = θ_k, keep stored p̂
```

> **Why valid:** Unbiased marginal likelihood estimates in MH ratio → valid asymptotically exact MCMC. This is **not** approximate — it targets the true posterior.

---

## 4.7 SIS Weight Recursion Derivation (2024 Q4a — always asked)

**Setup:** Markov proposal $q(x_{0:t}) = q(x_0)\prod_{k=1}^t q(x_k|x_{k-1})$.

Unnormalised weight: $\tilde{w}_t = \frac{p(x_{0:t},y_{1:t})}{q(x_{0:t})}$

**Sequential recursion:**
$$\tilde{w}_t = \tilde{w}_{t-1} \cdot \frac{f(x_t|x_{t-1})\,g(y_t|x_t)}{q(x_t|x_{t-1})}$$

**Proof:** Factor $p(x_{0:t},y_{1:t}) = p(x_{0:t-1},y_{1:t-1}) \cdot f(x_t|x_{t-1})\,g(y_t|x_t)$ and $q(x_{0:t}) = q(x_{0:t-1})\cdot q(x_t|x_{t-1})$ → cancel $t-1$ terms → incremental ratio remains ✓

---

## 4.8 Weight Degeneracy — Core Problem with SIS (2024)

**Theorem:** Relative variance grows **exponentially in $t$**.

For path targets $\pi(x_{0:t}) = \prod_{k=0}^t \mathcal{N}(x_k;0,1)$, proposal $q(x_{0:t}) = \prod_{k=0}^t \mathcal{N}(x_k;0,\sigma^2)$:

$$\frac{\text{Var}(\hat{Z}^N_t)}{Z_t^2} = \frac{1}{N}\left[\left(\frac{\sigma^4}{2\sigma^2-1}\right)^{t/2} - 1\right]$$

**Remedy:** Resampling at each step (SISR/BPF) — resets weights and eliminates degenerate particles.

---

## 4.9 Gradient of Log Marginal Likelihood (2025 Q4b)

$$\nabla_\theta \log p_\theta(y_{1:T}) = E_{p_\theta(x_{0:T}|y_{1:T})}[\nabla_\theta \log p_\theta(x_{0:T}, y_{1:T})]$$

**Proof:** Same log-derivative trick as Q2 gradient identity.

**Estimator + gradient ascent:**
$$\theta_{k+1} = \theta_k + \eta \cdot \frac{1}{N}\sum_{i=1}^N \nabla_\theta \log p_{\theta_k}(X_{0:T}^{(i)}, y_{1:T})$$

where $\{X_{0:T}^{(i)}\}$ come from the **SIR smoother** run on the BPF particles.

---

## 4.10 SIR Smoother (2025 Q4b)

**Goal:** Approximate the smoothing distribution $p_\theta(x_{0:T}|y_{1:T})$ — the full path distribution.

```
1. Run BPF forward: get particles {X_t^{(i)}} and weights {W_t^{(i)}} for t=0,...,T
2. At time T: resample full paths {X_{0:T}^{(i)}} according to final weights {W_T^{(i)}}
3. Output: N equally-weighted path particles {X_{0:T}^{(i)}} ≈ p(x_{0:T}|y_{1:T})
```

The resampling step in line 2 is exactly the SIR algorithm from Q2 cell §2.11.

---

## 4.11 Incremental Marginal Likelihood

$$p(y_t|y_{1:t-1}) = \int g(y_t|x_t)\,p(x_t|y_{1:t-1})\,dx_t, \quad p(y_{1:T}) = \prod_{t=1}^T p(y_t|y_{1:t-1})$$

# Cross-Cutting Patterns, Traps & Study Plan

---

## Recurring Patterns Across All Questions

### Pattern 1: The log-derivative trick (Q2 and Q4)

Every question involving $\nabla \log p_\theta(\cdot)$ uses the same three steps:
1. $\nabla \log p = \nabla p / p$
2. Swap integral and gradient (always stated as given)
3. $\nabla p_\theta(x,y) = p_\theta(x,y)\nabla\log p_\theta(x,y)$, then $p_\theta(x,y)/p_\theta(y) = p_\theta(x|y)$

### Pattern 2: Find optimal parameter via log-derivative (Q1, Q2, Q2 2023)

1. Write $\log M(\lambda)$ or $\log V(\lambda)$
2. $d\log M/d\lambda = 0$ → solve
3. Taking logs first removes products and simplifies algebra massively

### Pattern 3: IS/SNIS estimators — same weights, different formula

| Case | Target | Estimator | Biased? |
|------|--------|-----------|---------|
| IS (known Z) | $p = \bar{p}/Z$ | $\frac{1}{N}\sum w_i \varphi(X_i)$, $w_i = p(X_i)/q(X_i)$ | No |
| SNIS (unknown Z) | $\bar{p} \propto p$ | $\frac{\sum W_i \varphi(X_i)}{\sum W_i}$, $W_i = \bar{p}(X_i)/q(X_i)$ | Yes (consistent) |
| IS for $p(y_0)$ | marginal likelihood | $\frac{1}{N}\sum p(y_0\|X_i)\frac{p(X_i)}{q(X_i)}$, $X_i \sim q$ | No |

### Pattern 4: SIS → SISR → BPF progression

- **SIS**: no resampling → weights accumulate → exponential degeneracy
- **SISR**: SIS + resample at each step → stable weights → same structure each step  
- **BPF**: SISR with $q = f$ (transition as proposal) → $w_t^{(i)} = g(y_t|X_t^{(i)})$ only

### Pattern 5: Optimal IS = posterior

If you can find a proposal $q$ that matches the posterior $p(x|y) \propto p(y|x)p(x)$, IS variance = 0. When the proposal family can match the posterior exactly (e.g., 2023: Gaussian-Gaussian model), the optimal $\mu^*$ equals the posterior mean.

### Pattern 6: AR(1) stationarity (Q3 ULA analysis)

ULA with Gaussian target → AR(1). Stationary variance = $\sigma_q^2/(1-\rho^2)$ where $\rho = 1-\gamma/\sigma^2$.

---

## Common Exam Traps

| Trap | How to avoid |
|------|-------------|
| MH acceptance ratio: writing $p^*(x')/p^*(x)$ only | Always include $q$ terms: $\min(1, \frac{p^*(x')q(x\|x')}{p^*(x)q(x'\|x)})$ |
| ULA vs MALA confusion | ULA = no MH step (biased). MALA = Langevin proposal + MH accept/reject |
| SNIS for p(y): using only numerator | Denominator $\sum W_i$ estimates Z — both needed for SNIS |
| Constraint on $\lambda$ for rejection: forgetting it | Must state $\lambda < 1/2$ (or equivalent) — costs marks |
| IS variance formula: forgetting $-\bar{\varphi}^2$ | $\text{Var} = \frac{1}{N}(\int \frac{\varphi^2 p^2}{q}dx - \bar{\varphi}^2)$ |
| BPF: order of operations | Correct order: **Resample → Propagate → Weight → Normalise** |
| Weight degeneracy: saying variance grows polynomially | It grows **exponentially** in $t$ |
| Detailed balance proof: only showing one case | Must explicitly state Case 1 AND Case 2 (even if symmetric) |
| SNIS estimator: claiming it's unbiased | It's biased — say "biased but consistent" |
| Log marginal likelihood: normalising before extracting $w_t$ | Store unnormalised weights BEFORE normalisation |
| pMH: saying it's approximate | It's **exact** (asymptotically targets true posterior) — valid because BPF is unbiased |
| SISR vs SIS: mixing up resampling | SISR has resampling at each step; SIS does NOT |
| Gibbs sampler update order | Use the MOST RECENT value of each component — $x_n$ not $x_{n-1}$ in step 2 |

---

## Correct 5-Exam Map (ALL EXAMS NOW VERIFIED)

| File | Actual Year | Code | Type | Q1 | Q2 | Q3 | Q4 |
|------|------------|------|------|----|----|----|----|
| 2020.txt | 2020 | MATH97085 | ⚠️ OLD — SKIP | PRNGs, χ² test | Rejection/MH/inversion | MC integration | Gibbs |
| 2021.txt | 2021 | MATH97085 | ⚠️ OLD — SKIP | Congruential, Polar-Marsaglia | IS for tails, ratio-of-uniforms | MH, detailed balance | ARS/ARMS/Gibbs |
| 2022.txt | 2022 | MATH60047 | ⚠️ Trans. — mostly skip | Congruential, geometric, rejection | Cauchy/Normal, antithetic variates | IS, ratio-of-uniforms, Gibbs | Slice sampling |
| **2023.txt** | **2023** | **MATH70047** | ✅ CURRENT | Rayleigh CDF, inversion, rejection ($\mu^*=\sigma$) | MC for $p(y_0)$, IS variance, optimal IS ($\mu^*=y_0/2$) | Gibbs (3D), MH-within-Gibbs, MALA, IG-Gaussian | Filtering recursions, SISR→BPF, marginal likelihood, pMH |
| **2024.txt** | **2024** | **MATH70047** | ✅ CURRENT | Weibull, Box-Muller, Erlang rejection ($\mu^*=\lambda/k$) | Rayleigh IS, $\lambda^*=\sqrt{3/2}$, IS vs MC | MH+MALA+ULA+SG-ULA, Gamma-Poisson | Path SNIS, SIS derivation, variance degeneracy |
| **2025.txt** | **2025** | **MATH70047** | ✅ CURRENT | General transform, χ²-polar, rejection+Binomial ($\lambda^*=1/6$) | SNIS for $p(y)$, gradient identity, IS for indicator | Indep MH, general MH, detailed balance, SA+Langevin, ULA AR(1) | BPF, log marginal likelihood, gradient ascent + SIR smoother |

> **2020 and 2021 are MATH97085 (old curriculum)** — completely different topics (PRNGs, LCGs, chi-squared tests, antithetic variates, ARS/ARMS). None of these appear anywhere in the current lecture notes or 2023-2025 exams. **Safe to skip entirely.**

---

## What Appeared Only in 2023 (could return)

- **Gibbs sampler** (3D pseudocode, update order)
- **Metropolis-within-Gibbs** (for unnormalised conditionals)
- **IG-Gaussian conjugacy** (deriving $p(\sigma^2|x) = \text{IG}(\alpha+1/2, \beta+x^2/2)$)
- **Filtering recursions** (prediction + update equations written explicitly)
- **Particle Metropolis-Hastings** (pMH pseudocode + unbiasedness argument)
- **MCMC diagnostics** (thinning + ESS definition)
- **Rayleigh CDF proof by substitution** ($u = x^2$)
- **Gaussian proposal for rejection** (optimal $\mu^*=\sigma$ for Rayleigh target)

---

## Study Priority & Must-Memorise

> **Exam structure: 3 of 4 questions (20 marks each = 60 total).**  
> Target **Q1 + Q2 + Q3** — these are the most predictable and well-covered.  
> Q4 (SMC) is optional — only prepare it if you have spare time or want a fallback for Q3.

### Memorise cold (write 3× each):
1. **General MH pseudocode** + acceptance ratio
2. **MALA pseudocode** including proposal $q(x'|x)$
3. **ULA update rule** + know it's biased (no accept/reject)
4. **BPF pseudocode** (Resample → Propagate → Weight → Normalise)
5. **Detailed balance proof** (Case 1 + Case 2, ~8 lines)
6. **Gibbs sampler** (3D, use most-recent values)
7. **Particle MH pseudocode** + "valid because BPF is unbiased"
8. **ESS formula**: $\text{ESS}_N = 1/\sum_{i=1}^N \bar{w}_i^2$, range $[1,N]$

### Derive quickly (practice 2× each):
- IS variance formula + $-\bar{\varphi}^2$ term
- IG-Gaussian posterior conjugacy (combine exponents)
- AR(1) stationary variance → unbiased condition for ULA
- Filtering recursions (prediction + update)
- SIS weight recursion proof (factorisation)

# Old & Transitional Curriculum Content
## Files: 2020.txt (MATH97085), 2021.txt (MATH97085), 2022.txt (MATH60047/97085)

> These are **low-priority** for exam prep. Topics either don't appear in MATH70047 or have equivalents covered above.

---

## Module Code History

| Year | Code(s) | Curriculum |
|------|---------|-----------|
| 2020 | MATH97085 | OLD — PRNGs, rejection, MC basics, Gibbs |
| 2021 | MATH97085 | OLD — PRNGs, Pareto, Polar-Marsaglia, IS, MH, ARS/ARMS |
| 2022 | MATH60047/97085 | TRANSITIONAL — first year with new code, but old Q4 (slice sampling) |
| 2023+ | MATH70047 | CURRENT — no PRNGs, focuses on Langevin/ULA/pMH |

---

## Old Curriculum Topics (2020–2021)

### Pseudo-Random Number Generators (PRNGs)
- **Linear Congruential Generator:** $X_{n+1} = (aX_n + b)\mod m$
- Full period conditions: $\gcd(b,m)=1$, $a \equiv 1 \pmod{p}$ for all prime $p|m$, and $4|(a-1)$ if $4|m$
- Problems: correlations in low dimensions, short period if poorly chosen

### Polar-Marsaglia Method (2021 Q1)
- Generate two Normals without trigonometric functions
- Algorithm: accept $(U_1, U_2) \sim \text{Unif}(-1,1)^2$ if $S = U_1^2+U_2^2 < 1$, then $X_1 = U_1\sqrt{-2\log S/S}$, $X_2 = U_2\sqrt{-2\log S/S}$
- Advantage over Box-Muller: avoids $\sin/\cos$

### Ratio of Uniforms (2021, 2022)
- Enclose the set $C_h = \{(u,v): 0 \leq u \leq \sqrt{h(v/u)}\}$
- Sample $u,v$ from the bounding rectangle, accept if $(u,v) \in C_h$
- Works when $h$ is integrable and the bounding rectangle is finite

### Adaptive Rejection Sampling (ARS) (2021 Q4)
- For **log-concave** targets: construct piecewise exponential envelope from tangent lines to $\log p$
- Advantage: envelope tightens automatically as more points are sampled → acceptance rate → 1
- **ARMS** (Adaptive Rejection Metropolis Sampling): extends to non-log-concave by adding MH correction

---

## Transitional Topics (2022) — Potentially Relevant

### Antithetic Variates (2022 Q2b)
- Reduce variance using negatively correlated estimators
- If $\hat{\theta}_1 = h(U)$ and $\hat{\theta}_2 = h(1-U)$ with $h$ monotone: $\text{Var}(\frac{\hat{\theta}_1+\hat{\theta}_2}{2}) < \text{Var}(\hat{\theta}_1)$
- Classic example: $\hat{\theta}_1 = -\log(U_1)$, $\hat{\theta}_2 = -\log(1-U_1)$ as Exp(1) estimators

### Truncated Distribution Inversion (2022 Q2c)
- For $X$ truncated to $(a,b)$ with CDF $F_X$:
  - $F_{X|a<X<b}(x) = \frac{F_X(x) - F_X(a)}{F_X(b)-F_X(a)}$
  - Inversion: $X = F_X^{-1}(U(F_X(b)-F_X(a)) + F_X(a))$

### Slice Sampling (2022 Q4)
- **Advantage over MH/Gibbs:** No need to choose proposal bandwidth or know conditional distributions in closed form
- **Principle:** Augment target with auxiliary variable $u \sim \text{Unif}(0, p(x))$. Uniform distribution over $\{(x,u): u < p(x)\}$ has marginal $\propto p(x)$.
- **Gibbs on augmented space:** alternate sampling $u|x$ (trivial: $\text{Unif}(0,p(x))$) and $x|u$ (uniform on slice $\{x: p(x) > u\}$)
- **Main difficulty:** Computing the slice $\{x: p(x) > u\}$ may require bracketing/stepping-out procedures

### Bivariate Gaussian Gibbs (2022 Q3)
- For $Z = (X,Y)^\top \sim \mathcal{N}(\mu, \Sigma)$:
  - Full conditional: $X|Y \sim \mathcal{N}(\mu_X + \Sigma_{XY}\Sigma_{YY}^{-1}(Y-\mu_Y),\ \Sigma_{XX}-\Sigma_{XY}\Sigma_{YY}^{-1}\Sigma_{YX})$
  - Gibbs sampler alternates between these two 1D Gaussians

> The 2023 MATH70047 exam also includes Gibbs (3D) — see Q3 section above for the current curriculum version.